In [1]:
import csv
import os
import pandas as pd
import re
import shutil
import time

from collections import defaultdict, deque, Counter
from dotenv import load_dotenv
from typing import List
from pydantic import BaseModel, Field

from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

from google import genai


# Load environment variables
load_dotenv(override=True)

MODEL= os.getenv("MODEL")
MODEL_SUPERVISOR= os.getenv("MODEL_SUPERVISOR")
MODEL_EMBEDDING = os.getenv("MODEL_EMBEDDING")
API_KEY = os.getenv("GEMINI_API_KEY")

client = genai.Client(api_key=API_KEY)

BASE_DIR = os.path.dirname(os.getcwd())
DOCUMENT_FOLDER = os.path.join(BASE_DIR, "data/full_text_pdf_rag")
VECTOR_DB_PATH = os.path.join(BASE_DIR, "vectorstore_sensitivity", "pdf_index")

# Remove previous FAISS index if present
if os.path.exists(VECTOR_DB_PATH):
    print(f"[CLEANUP] Removed existent vectorstore: {VECTOR_DB_PATH}")
    shutil.rmtree(VECTOR_DB_PATH)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [2]:
# Embedding and text split configuration

CHUNK_SIZE = 2000
CHUNK_OVERLAP = 400
MAX_CONTEXT_CHARS = 30000

# Simple rate-limiting mechanism
_RATE_WINDOW_SEC = 60.0
_RATE_MAX_CALLS = 10
_rate_calls_log = deque()

_sent_calls_count = 0
_total_input_tokens = 0
_total_output_tokens = 0
_total_embedding_tokens = 0


def _rate_limit_block_until_allowed():
    """Blocks execution to respect Gemini API rate limits."""
    while True:
        now = time.monotonic()

        while _rate_calls_log and (now - _rate_calls_log[0]) >= _RATE_WINDOW_SEC:
            _rate_calls_log.popleft()

        if len(_rate_calls_log) < _RATE_MAX_CALLS:
            _rate_calls_log.append(now)
            return

        time.sleep(0.5)


# Pydantic response schemas
class ExtractedDatasets(BaseModel):
    datasets: List[str] = Field(description="List of official AQA dataset names found")

class HarmonizedSources(BaseModel):
    sources: List[str] = Field(description="Harmonized list of dataset names in input order")


def truncate_context(text: str, limit_chars: int) -> str:
    """Truncates long text while preserving character encoding."""
    if len(text) <= limit_chars:
        return text
    return text[:limit_chars - 3] + "..."

def clean_text(text: str) -> str:
    """Removes invalid UTF-8 characters."""
    return text.encode("utf-8", "ignore").decode("utf-8", "ignore")


# FAISS Index Construction
def build_or_load_index(document_folder: str, vector_db_path: str):
    """Builds or loads the FAISS index for PDF documents."""
    embedding_model = GoogleGenerativeAIEmbeddings(model=MODEL_EMBEDDING, google_api_key=API_KEY)

    if os.path.exists(vector_db_path):
        try:
            vs = FAISS.load_local(vector_db_path, embedding_model, allow_dangerous_deserialization=False)
            print(f"[FAISS] Loaded index from {vector_db_path}. Total vectors: {vs.index.ntotal}")
            return vs, vs.as_retriever(), vs.index.ntotal
        except Exception as e:
            print(f"[FAISS] Error loading index ({e}), rebuilding index.")

    # Load and chunk PDFs
    pdf_files = [f for f in os.listdir(document_folder) if f.endswith(".pdf")]
    all_docs = []

    for file in pdf_files:
        path = os.path.join(document_folder, file)
        loader = PyPDFLoader(path)
        pages = loader.load()

        print(f"[LOAD] {file}: {len(pages)} pages")

        for p in pages:
            p.metadata["source"] = file

        all_docs.extend(pages)

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    documents = text_splitter.split_documents(all_docs)

    # Clean encoding issues
    for d in documents:
        try:
            d.page_content.encode("utf-8")
        except UnicodeEncodeError as e:
            print(f"[WARN] Unicode error in {d.metadata.get('source')}: {e}")
        d.page_content = clean_text(d.page_content)

    global _total_embedding_tokens
    all_chunk_texts = [d.page_content for d in documents]
    count_resp = client.models.count_tokens(
        model=MODEL_EMBEDDING,
        contents=all_chunk_texts
    )
    _total_embedding_tokens = count_resp.total_tokens

    # Show chunk distribution per PDF
    counter = Counter(d.metadata['source'] for d in documents)
    for pdf_name, num_chunks in counter.items():
        print(f"[SPLIT] {pdf_name}: {num_chunks} chunks created")

    # Build FAISS index
    vectorstore = FAISS.from_documents(documents, embedding_model)
    total_chunks = vectorstore.index.ntotal

    print(f"[FAISS] Created vectors: {total_chunks}")

    os.makedirs(vector_db_path, exist_ok=True)
    vectorstore.save_local(vector_db_path)

    print(f"[FAISS] Saved index to {vector_db_path}")
    return vectorstore, vectorstore.as_retriever(), total_chunks


# Dataset extraction
def extract_datasets_per_pdf(retriever, query: str, total_chunks: int = 100, top_chunks_per_paper: int = 5):
    """Retrieves chunks via RAG and extracts AQA dataset names via LLM."""
    global _sent_calls_count, _total_input_tokens, _total_output_tokens

    k = total_chunks // 3
    fetch_k = total_chunks // 2

    results = retriever.vectorstore.max_marginal_relevance_search(
        query, k=k, fetch_k=fetch_k, lambda_mult=0.9
    )

    print(f"[RAG] Retrieved results: {len(results)} chunks")

    # Group chunks per PDF
    groups = defaultdict(list)
    for doc in results:
        src = doc.metadata.get("source", "Unknown")
        groups[src].append(doc.page_content)

    paper_datasets = {}

    # Process each PDF
    for source, chunks in groups.items():
        selected_chunks = chunks[:top_chunks_per_paper]
        combined = "\n\n---\n\n".join(selected_chunks)
        context = truncate_context(combined, MAX_CONTEXT_CHARS)

        prompt = (
            "You are an assistant that analyzes full scientific papers.\n"
            "Given the following extracted text from a paper, extract ONLY the official and recognized names of datasets that satisfy the following conditions:\n"
            "(a) The dataset is specifically used for Human Action Quality Assessment (AQA). This condition must be strictly respected.\n"
            "(b) Include only real, unique, and identifiable resources (dataset, database, or data collection).\n"
            "(c) Do NOT write generic descriptions (e.g., 'a large dataset', 'clinical dataset'); if the paper only describes the data generically, leave this field empty.\n\n"
            f"Text:\n{context}\n\n"
            "Output Instructions:\n"
            "- Return valid JSON only, with a single field 'datasets' that is a list of dataset names, e.g., {\"datasets\": [\"FINA09\",\"MTL-AQA\"]}.\n"
            "- If multiple datasets are mentioned, include all of them in the same list.\n"
            "- If no datasets meet these conditions, return an empty list.\n"
        )

        _rate_limit_block_until_allowed()

        backoff = 5
        extracted = []

        # Retry logic
        for attempt in range(4):
            try:
                _sent_calls_count += 1
                resp = client.models.generate_content(
                    model=MODEL,
                    contents=prompt,
                    config={
                        "response_mime_type": "application/json",
                        "response_schema": ExtractedDatasets,
                        "temperature": 0.0,
                        "top_k": 1,
                        "top_p": 1.0
                    },
                )

                # Token tracking
                if resp.usage_metadata:
                    _total_input_tokens += resp.usage_metadata.prompt_token_count
                    _total_output_tokens += resp.usage_metadata.candidates_token_count

                extracted = resp.parsed.datasets or []
                extracted = [s.strip() for s in extracted if s and s.strip()]
                break

            except Exception as e:
                print(f"[LLM ERROR] paper={source} attempt={attempt+1} error={e}")
                time.sleep(backoff)
                backoff *= 2

        paper_datasets[source] = extracted
        print(f"[EXTRACT] {source}: found {len(extracted)} datasets")

    # Prepare output table
    all_pdf_files = [f for f in os.listdir(DOCUMENT_FOLDER) if f.endswith(".pdf")]

    final_list = []
    for pdf in all_pdf_files:
        datasets = paper_datasets.get(pdf, [])
        datasets_str = ";".join(sorted(set(datasets))) if datasets else ""
        final_list.append({"Title": pdf, "Data Source": datasets_str})

    return final_list


# Execution
start_index_time = time.monotonic()
vectorstore, retriever, total_chunks = build_or_load_index(DOCUMENT_FOLDER, VECTOR_DB_PATH)
end_index_time = time.monotonic()
total_index_time_seconds = end_index_time - start_index_time

query = "Human Action Quality Assessment (AQA) datasets"

start_total_time = time.monotonic()
extracted = extract_datasets_per_pdf(retriever, query, total_chunks=total_chunks, top_chunks_per_paper=5)
end_total_time = time.monotonic()
total_extract_time_seconds = end_total_time - start_total_time

total_pipeline_time_seconds = total_index_time_seconds + total_extract_time_seconds

df_results = pd.DataFrame(extracted)
print("[DONE] Extraction completed.")

[LOAD] Baby et al. - 2022 - Representation Learning Using Rank Loss for Robust Neurosurgical Skills Evaluation.pdf: 5 pages
[LOAD] Chang - 2022 - Research on Tennis Motion Evaluation Method and System Based on Deep Learning.pdf: 13 pages
[LOAD] Chen et al. - 2024 - Auto-encoding score distribution regression for action quality assessment.pdf: 14 pages
[LOAD] Dadashzadeh et al. - 2024 - PECoP Parameter Efficient Continual Pretraining for Action Quality Assessment.pdf: 11 pages
[LOAD] Du et al. - 2023 - View-Invariant Center-of-Pressure Metrics Estimation With Monocular RGB Camera.pdf: 14 pages
[LOAD] Du et al. - 2024 - Learning Semantics-Guided Representations for Scoring Figure Skating.pdf: 11 pages
[LOAD] Fang et al. - 2024 - Which is the Better Teacher Action A New Ranking Model and Dataset.pdf: 5 pages
[LOAD] Farabi et al. - 2022 - Improving Action Quality Assessment Using Weighted Aggregation.pdf: 12 pages
[LOAD] Gallardo et al. - 2024 - GYMetricPose A light-weight angle-based grap

In [3]:
# --- FINAL REPORT AND COSTS ---

theoretical_min_time = ((_sent_calls_count - 1) // _RATE_MAX_CALLS) * _RATE_WINDOW_SEC if _sent_calls_count > 0 else 0

print("\n" + "="*40)
print("EXECUTION REPORT - DATASET EXTRACTION")
print("="*40)

print(f"Papers Processed: {len(extracted)}")
print(f"API Calls made: {_sent_calls_count}")
print(f"Input Tokens consumed: {_total_input_tokens}")
print(f"Output Tokens generated: {_total_output_tokens}")
print(f"Embedding Tokens: {_total_embedding_tokens}")
print(f"Execution Time: {total_pipeline_time_seconds/60:.2f} min (Theoretical: {theoretical_min_time/60:.2f} min)")

# Pricing
PRICE_PER_1M_INPUT_TOKENS = 0.3
PRICE_PER_1M_OUTPUT_TOKENS = 2.5
PRICE_PER_1M_EMBEDDING_TOKENS = 0.20

cost_input = (_total_input_tokens / 1_000_000) * PRICE_PER_1M_INPUT_TOKENS
cost_output = (_total_output_tokens / 1_000_000) * PRICE_PER_1M_OUTPUT_TOKENS
cost_embedding = (_total_embedding_tokens / 1_000_000) * PRICE_PER_1M_EMBEDDING_TOKENS
cost_total = cost_input + cost_output + cost_embedding

print("-" * 40)
print(f"Input Cost:  ${cost_input:.5f}")
print(f"Output Cost: ${cost_output:.5f}")
print(f"Embedding Cost: ${cost_embedding:.5f}")
print(f"Estimated Total Cost: ${cost_total:.5f}")
print("="*40)


EXECUTION REPORT - DATASET EXTRACTION
Papers Processed: 64
API Calls made: 64
Input Tokens consumed: 145344
Output Tokens generated: 1185
Embedding Tokens: 1014920
Execution Time: 10.32 min (Theoretical: 6.00 min)
----------------------------------------
Input Cost:  $0.04360
Output Cost: $0.00296
Embedding Cost: $0.20298
Estimated Total Cost: $0.24955


In [4]:
folder_path = '../data/'
result_path = folder_path + 'results/'
ELIGIBILITY_DIR = result_path + 'eligibility_sensitivity'

def append_run_summary(filepath, row_dict):
    file_exists = os.path.isfile(filepath)
    with open(filepath, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=row_dict.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row_dict)

append_run_summary(
    f"{ELIGIBILITY_DIR}/runs_cost_summary.csv",
    {
        "stage": "eligibility",
        "model": MODEL,
        "records_processed": len(df_results),
        "api_calls": _sent_calls_count,
        "input_tokens": _total_input_tokens,
        "output_tokens": _total_output_tokens,
        "embedding_tokens": _total_embedding_tokens,
        "index_time_sec": total_index_time_seconds,
        "extraction_time_sec": total_extract_time_seconds,
        "execution_time_sec": total_pipeline_time_seconds,
        "cost_input_usd": cost_input,
        "cost_output_usd": cost_output,
        "cost_embedding_usd": cost_embedding,
        "cost_usd": cost_total,
    }
)

In [5]:
# Harmonization

_harm_calls_count = 0
_harm_input_tokens = 0
_harm_output_tokens = 0

def harmonize_all_data_sources(df: pd.DataFrame, client, model=None, max_retries: int = 3):
    """Harmonizes dataset names across all rows using an LLM."""
    global _harm_calls_count, _harm_input_tokens, _harm_output_tokens

    if model is None:
        model = MODEL

    # Prepare records for the harmonization prompt
    records = []
    for idx, row in df.iterrows():
        sources = "" if pd.isna(row.get("Data Source")) else str(row.get("Data Source"))
        records.append(f"- id: {int(idx)}\n  Data Source: {sources}")

    prompt = (
        "Check and harmonize all dataset names for consistency and correctness.\n"
        "Guidelines:\n"
        "- Remove descriptors like 'dataset', 'database', 'data'.\n"
        "- If multiple datasets are listed, leave the separation with ';' (semicolon) without spaces between the datasets.\n"
        "- Always capitalize the first letter of each dataset.\n"
        "- Prefer the recognized abbreviation if it exists, otherwise use the full name (e.g., 'Massachusetts Institute of Technology' becomes 'MIT','Piano Skills Assessment' becomes 'PISA').\n"
        "- If a field is empty, keep it empty.\n\n"
        "Output Instructions:\n"
        "- Return only JSON matching the schema.\n"
        "- Return exactly one harmonized string per input row, in the same order as provided.\n\n"
        "Records to harmonize:\n"
        + "\n".join(records)
    )

    # Retry mechanism for robustness
    backoff = 5
    for attempt in range(max_retries):
        _rate_limit_block_until_allowed()
        try:
            _harm_calls_count += 1
            resp = client.models.generate_content(
                model=model,
                contents=prompt,
                config={
                    "response_mime_type": "application/json",
                    "response_schema": HarmonizedSources,
                    "temperature": 0.0,
                    "top_k": 1,
                    "top_p": 1.0
                },
            )
            if resp.usage_metadata:
                _harm_input_tokens += resp.usage_metadata.prompt_token_count
                _harm_output_tokens += resp.usage_metadata.candidates_token_count

            harmonized = resp.parsed.sources

            # Ensure output length matches input length
            if len(harmonized) != len(df):
                harmonized = (harmonized + [""] * len(df))[:len(df)]
            return harmonized

        except Exception as e:
            print(f"[HARMONIZE ERROR] attempt={attempt+1} error={e}")
            time.sleep(backoff)
            backoff *= 2

    # Fallback: return original values unchanged
    fallback = []
    for _, row in df.iterrows():
        s = "" if pd.isna(row.get("Data Source")) else str(row.get("Data Source"))
        fallback.append(s)

    return fallback


# Apply harmonization to dataframe
start_harm_time = time.monotonic()
df_results["Data Source Harmonized"] = harmonize_all_data_sources(df_results, client, model=MODEL_SUPERVISOR)
end_harm_time = time.monotonic()
total_harm_time_seconds = end_harm_time - start_harm_time

df_results

,Title,Data Source,Data Source Harmonized
0,Baby et al. - 2022 - Representation Learning U...,JIGSAWS;NETS,JIGSAWS;NETS
1,Chang - 2022 - Research on Tennis Motion Evalu...,,
2,Chen et al. - 2024 - Auto-encoding score distr...,AQA-7;JIGSAWS;MTL-AQA,AQA-7;JIGSAWS;MTL-AQA
3,Dadashzadeh et al. - 2024 - PECoP Parameter Ef...,FineDiving;JIGSAWS;MTL-AQA;PD4T,FineDiving;JIGSAWS;MTL-AQA;PD4T
4,Du et al. - 2023 - View-Invariant Center-of-Pr...,A data set with kinematic and ground reaction ...,Human Balance;Novel Body Landmark
...,...,...,...
59,Zhang et al. - 2023 - Label-reconstruction-bas...,,
60,Zhang et al. - 2023 - LOGO A Long-Form Video D...,AQA-7-Dive;FSD-10;FineDiving;LOGO;MTL Dive;MTL...,AQA-7-Dive;FSD-10;FineDiving;LOGO;MTL-Dive;MTL...
61,Zhang et al. - 2024 - Narrative Action Evaluat...,FineGym;MTL-AQA,FineGym;MTL-AQA
62,Zhou et al. - 2023 - A Video-Based Augmented R...,JDM dataset,JDM


In [6]:
# --- FINAL REPORT AND COSTS (HARMONIZATION) ---

print("\n" + "="*40)
print("EXECUTION REPORT - HARMONIZATION")
print("="*40)

print(f"API Calls made: {_harm_calls_count}")
print(f"Input Tokens consumed: {_harm_input_tokens}")
print(f"Output Tokens generated: {_harm_output_tokens}")

# Pricing GEMINI 2.5 Pro
PRICE_PER_1M_INPUT_TOKENS = 1.25
PRICE_PER_1M_OUTPUT_TOKENS = 10

cost_input = (_harm_input_tokens / 1_000_000) * PRICE_PER_1M_INPUT_TOKENS
cost_output = (_harm_output_tokens / 1_000_000) * PRICE_PER_1M_OUTPUT_TOKENS
cost_total = cost_input + cost_output

print("-" * 40)
print(f"Input Cost:  ${cost_input:.5f}")
print(f"Output Cost: ${cost_output:.5f}")
print(f"Estimated Total Cost: ${cost_total:.5f}")
print("="*40)


EXECUTION REPORT - HARMONIZATION
API Calls made: 1
Input Tokens consumed: 1674
Output Tokens generated: 945
----------------------------------------
Input Cost:  $0.00209
Output Cost: $0.00945
Estimated Total Cost: $0.01154


In [7]:
append_run_summary(
    f"{ELIGIBILITY_DIR}/runs_cost_summary.csv",
    {
        "stage": "harmonization",
        "model": MODEL_SUPERVISOR,
        "records_processed": len(df_results),
        "api_calls": _harm_calls_count,
        "input_tokens": _harm_input_tokens,
        "output_tokens": _harm_output_tokens,
        "embedding_tokens": None,
        "index_time_sec": None,
        "extraction_time_sec": None,
        "execution_time_sec": total_harm_time_seconds,
        "cost_input_usd": cost_input,
        "cost_output_usd": cost_output,
        "cost_embedding_usd": None,
        "cost_usd": cost_total,
    }
)

In [8]:
df_results.to_csv(f"{ELIGIBILITY_DIR}/records.csv", index=False)

In [9]:
# Final DataFrame Preparation
df_evaluated_final = df_results.copy()
df_evaluated_final["Data Source"] = df_evaluated_final["Data Source Harmonized"]
df_evaluated_final.drop(columns=["Data Source Harmonized"], inplace=True)

# Clean and normalize the Title field
df_evaluated_final["Title"] = (
    df_evaluated_final["Title"]
    .astype(str)
    .str.replace(r"^.* - ", "", regex=True)
    .str.replace(r"\.pdf$", "", flags=re.IGNORECASE, regex=True)
    .str.strip()
)
df_evaluated_final

,Title,Data Source
0,Representation Learning Using Rank Loss for Ro...,JIGSAWS;NETS
1,Research on Tennis Motion Evaluation Method an...,
2,Auto-encoding score distribution regression fo...,AQA-7;JIGSAWS;MTL-AQA
3,PECoP Parameter Efficient Continual Pretrainin...,FineDiving;JIGSAWS;MTL-AQA;PD4T
4,View-Invariant Center-of-Pressure Metrics Esti...,Human Balance;Novel Body Landmark
...,...,...
59,Label-reconstruction-based pseudo-subscore lea...,
60,LOGO A Long-Form Video Dataset for Group Actio...,AQA-7-Dive;FSD-10;FineDiving;LOGO;MTL-Dive;MTL...
61,Narrative Action Evaluation with Prompt-Guided...,FineGym;MTL-AQA
62,A Video-Based Augmented Reality System for Hum...,JDM


In [10]:
# Merge abstract metadata with the harmonized dataset sources
folder_path = '../data/'
result_path = folder_path + 'results/'

df_abstract = pd.read_csv(result_path + 'df_evaluated_validation.csv')

# Sort both tables by Title for proper alignment
df_abstract_sorted = df_abstract.sort_values(by="Title").reset_index(drop=True)
df_evaluated_final_sorted = df_evaluated_final.sort_values(by="Title").reset_index(drop=True)
df_abstract_sorted

,DOI,Title,Authors,Year,Abstract,Include,Reason,Data Source
0,10.1109/TNSRE.2023.3317411,A Contrastive Learning Network for Performance...,L. Yao; Q. Lei; H. Zhang; J. Du; S. Gao,2023,Human activity analysis in the legal monitorin...,yes,The paper proposes a contrastive learning fram...,UI-PRMD;KIMORE
1,10.1155/2022/8950732,A Data Mining-Based Model for Evaluating Tenni...,"Chen, Hang",2022,This paper uses data mining technology to math...,yes,The paper uses vision-based methods (RGBD imag...,NaN
2,10.1145/3581783.3613774,A Figure Skating Jumping Dataset for Replay-Gu...,"Liu, Y.; Cheng, X.; Ikenaga, T.",2023,"In competitive sports, judges often scrutinize...",yes,The paper constructs and uses a new dataset (R...,RFSJ
3,10.1007/978-3-031-44216-2_19,A Graph Convolutional Siamese Network for the ...,"Li, C.; Ling, X.; Xia, S.",2023,"Recently, due to the attention of physical reh...",yes,The paper uses vision-based methods (3D skelet...,UI-PRMD;IntelliRehabDS
4,10.1109/VCIP53242.2021.9675430,A Multi-dimensional Aesthetic Quality Assessme...,T. Wang; W. Sun; X. Min; W. Lu; Z. Zhang; G. Zhai,2021,With the development of the game industry and ...,no,The paper is about Aesthetic Quality Assessmen...,NaN
...,...,...,...,...,...,...,...,...
140,10.1109/ICCT.2015.7399879,Virtual Personal Trainer via the Kinect Sensor,Xin Jin; Yuan Yao; Qiliang Jiang; Xingying Hua...,2015,Our daily life is more convenient than ever be...,yes,The paper proposes a vision-based system for r...,NaN
141,10.1109/CVPR.2019.00039,What and How Well You Performed? A Multitask L...,P. Parmar; B. T. Morris,2019,Can performance on the task of action quality ...,yes,The paper proposes a multitask learning approa...,Multitask-AQA
142,10.1016/j.jvcir.2021.103304,What and how well you exercised? An efficient ...,"Li, J.; Hu, Q.; Guo, T.; Wang, S.; Shen, Y.",2021,Human action analysis has been an active resea...,yes,The paper proposes a framework to assess fitne...,Fitness-28
143,10.1109/ICASSP48485.2024.10448158,Which is the Better Teacher Action? A New Rank...,M. Fang; X. Du; Q. Liu; Y. Zhou; Q. Liang; S. Liu,2024,"Teachers as leaders of classroom teaching, can...",yes,The paper introduces a new dataset (TAQR) for ...,TAQR


In [11]:
df_subset = df_abstract_sorted[
    df_abstract_sorted["Include"].str.lower().isin(["yes", "maybe"])
    & df_abstract_sorted["Data Source"].notna()
].copy()

df_subset

,DOI,Title,Authors,Year,Abstract,Include,Reason,Data Source
0,10.1109/TNSRE.2023.3317411,A Contrastive Learning Network for Performance...,L. Yao; Q. Lei; H. Zhang; J. Du; S. Gao,2023,Human activity analysis in the legal monitorin...,yes,The paper proposes a contrastive learning fram...,UI-PRMD;KIMORE
2,10.1145/3581783.3613774,A Figure Skating Jumping Dataset for Replay-Gu...,"Liu, Y.; Cheng, X.; Ikenaga, T.",2023,"In competitive sports, judges often scrutinize...",yes,The paper constructs and uses a new dataset (R...,RFSJ
3,10.1007/978-3-031-44216-2_19,A Graph Convolutional Siamese Network for the ...,"Li, C.; Ling, X.; Xia, S.",2023,"Recently, due to the attention of physical reh...",yes,The paper uses vision-based methods (3D skelet...,UI-PRMD;IntelliRehabDS
6,10.1155/2023/3649217,A Novel Model for Intelligent Pull-Ups Test Ba...,"Liu, G.; Wang, J.; Zhang, Z.; Liu, Q.; Ren, Y....",2023,Applying computer vision and machine learning ...,yes,The paper uses computer vision for intelligent...,Self-Produced Pull-Ups
9,10.1109/EIECC60864.2023.10456639,A Temporal Action Evaluation Algorithm for Med...,Z. Wang; J. Ruan; J. Zhang; J. Yue,2023,The introduction of temporal action detection ...,yes,The paper proposes an algorithm for temporal a...,Thumos14;ActivityNet1.3;Self-Made Obstetric Pa...
...,...,...,...,...,...,...,...,...
138,10.1145/3606038.3616150,Video-based Skill Assessment for Golf: Estimat...,"Ingwersen, C.K.; Xarles, A.; Clapés, A.; Madad...",2023,Automated skill assessment in sports using vid...,yes,The paper focuses on video-based skill assessm...,Golf Swing Video
139,10.1109/TMM.2022.3222681,View-Invariant Center-of-Pressure Metrics Esti...,C. Du; S. Graham; C. Depp; T. Nguyen,2023,"Center of pressure (CoP) metrics, including Co...",yes,The paper proposes a vision-based framework fo...,Multi-View Body Landmark
141,10.1109/CVPR.2019.00039,What and How Well You Performed? A Multitask L...,P. Parmar; B. T. Morris,2019,Can performance on the task of action quality ...,yes,The paper proposes a multitask learning approa...,Multitask-AQA
142,10.1016/j.jvcir.2021.103304,What and how well you exercised? An efficient ...,"Li, J.; Hu, Q.; Guo, T.; Wang, S.; Shen, Y.",2021,Human action analysis has been an active resea...,yes,The paper proposes a framework to assess fitne...,Fitness-28


In [12]:
df_subset_clean = df_subset.copy()
df_subset_clean["Title_key"] = df_subset_clean["Title"].astype(str).str.replace(":", "", regex=False).str.replace("?", "", regex=False).str.lower().str[:35]

df_evaluated_final_clean = df_evaluated_final.copy()
df_evaluated_final_clean["Title_key"] = df_evaluated_final_clean["Title"].astype(str).str.lower().str[:35]

df_data_sources = (
    df_subset_clean[["DOI", "Title_key", "Data Source"]]
    .rename(columns={"Data Source": "Data Source Screening"})
    .merge(
        df_evaluated_final_clean[["Title", "Title_key", "Data Source"]]
        .rename(columns={"Data Source": "Data Source Sensitivity"}),
        on="Title_key",
        how="inner"
    )
    .drop(columns="Title_key")
)

df_data_sources

,DOI,Data Source Screening,Title,Data Source Sensitivity
0,10.1109/TNSRE.2023.3317411,UI-PRMD;KIMORE,A Contrastive Learning Network for Performance...,KIMORE;UI-PRMD
1,10.1145/3581783.3613774,RFSJ,A Figure Skating Jumping Dataset for Replay-Gu...,AQA-7;BPAD;FR-FS;FineDiving;Fis-V;Fitness-AQA;...
2,10.1007/978-3-031-44216-2_19,UI-PRMD;IntelliRehabDS,A Graph Convolutional Siamese Network for the ...,
3,10.1155/2023/3649217,Self-Produced Pull-Ups,A Novel Model for Intelligent Pull-Ups Test Ba...,SDUST-PUT
4,10.1109/EIECC60864.2023.10456639,Thumos14;ActivityNet1.3;Self-Made Obstetric Pa...,A Temporal Action Evaluation Algorithm for Med...,
...,...,...,...,...
59,10.1145/3606038.3616150,Golf Swing Video,Video-based Skill Assessment for Golf Estimati...,
60,10.1109/TMM.2022.3222681,Multi-View Body Landmark,View-Invariant Center-of-Pressure Metrics Esti...,Human Balance;Novel Body Landmark
61,10.1109/CVPR.2019.00039,Multitask-AQA,What and How Well You Performed A Multitask Le...,MIT-Dive;MTL-AQA;UNLV-Dive
62,10.1016/j.jvcir.2021.103304,Fitness-28,What and how well you exercised An efficient a...,Fitness-28


In [13]:
def sort_semicolon_strings(x):
    if pd.isna(x):
        return x
    return "; ".join(sorted([s.strip() for s in str(x).split(";") if s.strip()]))

df_data_sources["Data Source Screening"] = df_data_sources["Data Source Screening"].apply(sort_semicolon_strings)
df_data_sources["Data Source Sensitivity"] = df_data_sources["Data Source Sensitivity"].apply(sort_semicolon_strings)
df_data_sources = df_data_sources.drop(columns="Title")

df_data_sources

,DOI,Data Source Screening,Data Source Sensitivity
0,10.1109/TNSRE.2023.3317411,KIMORE; UI-PRMD,KIMORE; UI-PRMD
1,10.1145/3581783.3613774,RFSJ,AQA-7; BPAD; FR-FS; FineDiving; Fis-V; Fitness...
2,10.1007/978-3-031-44216-2_19,IntelliRehabDS; UI-PRMD,
3,10.1155/2023/3649217,Self-Produced Pull-Ups,SDUST-PUT
4,10.1109/EIECC60864.2023.10456639,ActivityNet1.3; Self-Made Cardiopulmonary Resu...,
...,...,...,...
59,10.1145/3606038.3616150,Golf Swing Video,
60,10.1109/TMM.2022.3222681,Multi-View Body Landmark,Human Balance; Novel Body Landmark
61,10.1109/CVPR.2019.00039,Multitask-AQA,MIT-Dive; MTL-AQA; UNLV-Dive
62,10.1016/j.jvcir.2021.103304,Fitness-28,Fitness-28


In [14]:
df_mismatch = df_data_sources[
    df_data_sources["Data Source Sensitivity"].notna()
    & (df_data_sources["Data Source Sensitivity"].astype(str).str.strip() != "")
    & (
        df_data_sources["Data Source Screening"]
        != df_data_sources["Data Source Sensitivity"]
    )
].copy()

df_mismatch

,DOI,Data Source Screening,Data Source Sensitivity
1,10.1145/3581783.3613774,RFSJ,AQA-7; BPAD; FR-FS; FineDiving; Fis-V; Fitness...
3,10.1155/2023/3649217,Self-Produced Pull-Ups,SDUST-PUT
5,10.1109/TVCG.2023.3247092,3D Animation; JDM,JDM
6,10.1109/WACV.2019.00161,AQA,AQA-7
9,10.1007/978-3-319-10599-4_36,Olympic Sports,Diving; Figure Skating
10,10.1007/s00521-023-09068-w,AQA-7; JIGSAWSS; MTL-AQA,AQA-7; JIGSAWS; MTL-AQA
12,10.1007/978-3-031-19839-7_7,Fitness-AQA,Fitness-AQA; Waseda Backsquat
13,10.1109/TPAMI.2024.3378753,EHE; KIMORE; UI-PRMD,EHE; KIMORE
14,10.1109/TIP.2023.3331212,AQA-7; FineDiving; MTL-AQA,AQA-7; AQA-7-Dive; FineDiving; MIT-Dive; MTL-A...
15,10.1109/CVPR52688.2022.00296,FineDiving,FineDiving; First-Person Basketball; MIT-Skate...


In [38]:
# Map partial dataset names to standardized identifiers
dataset_map = {
    "AQA": ["AQA-7"],
    "MIT-Skating": ["MIT-Skate"],
    "MIT-Diving": ["MIT-Dive"],
    "MIT-Skiing": ["MIT-Skate"],
    "Olympics Diving": ["MIT-Dive"],
    "MIT Olympic Scoring": ["MIT-Skate", "MIT-Dive"],
    "Olympic Sports": ["MIT-Dive", "MIT-Skate"],
    "UNLV-Skating": ["UNLV-Skate"],
    "UNLV-Diving": ["UNLV-Dive"],
    "Vault": ["UNLV-Vault"],
    "Gymnastic vaults": ["UNLV-Vault"],
    "UNLV Olympic Scoring": ["UNLV-Skate", "UNLV-Dive", "UNLV-Vault"],
    "FineDiving-HM": ["FineDiving"],
    "Fis-V": ["FIS-V"],
    "FisV-5": ["FIS-V"],
    "IntelliRehabDS": ["IRDS"],
    "Multitask-AQA": ["MTL-AQA"],
    "Fitness Action": ["Fitness-AQA"],
    "FitnessAQA": ["Fitness-AQA"],
    "RG": ["Rhythmic Gymnastics"],
    "MLT-AQA": ["MTL-AQA"],
    "AFG-Olympics": ["AGF-Olympics"],
    "Epic Skills 2018": ["EPIC-Skills"],
    "MIT-Diving & Skiing": ["MIT-Dive"],
    "Kimore": ["KIMORE"],
    "G-ExPose": ["ExPose"],

    "MIT Figure Skating": ["MIT-Skate"],
    "Figure Skating": ["MIT-Skate"],
    "JIGSAWSS": ["JIGSAWS"],
    "MSR Action 3D": ["MSR-Action3D"],
    "MSRC12": ["MSRC-12"],
    "Play Therapy 13 (PT13)": ["PT13"],

    "RhythmicGymnastics": ["Rhythmic Gymnastics"],
    "Diving": ["MIT-Dive"],
    "SDUST-PUT": [""],
    "FigureSkating": ["MIT-Skate"],
    "OrigamiVideo": ["Origami Video"],
    "Egohands;": [""],
    "CVD; ": [""],
    ";FR-FS;": [""],
    "Olympic Judging": ["MIT-Skate", "MIT-Dive"]
}

def map_partial_sources(cell):
    """Replace partial or alternative dataset names with standardized identifiers."""
    if pd.isna(cell):
        return cell

    # Split multiple datasets by semicolon
    parts = [p.strip() for p in str(cell).split(";") if p.strip()]
    out = []

    for p in parts:
        matched = False
        for k, v in dataset_map.items():
            pattern = fr"(^|;){re.escape(k)}($|;)"
            if re.fullmatch(k, p):
                out.extend(v)
                matched = True
                break
            elif re.search(pattern, ";" + p + ";"):
                out.extend(v)
                matched = True
                break
        if not matched:
            out.append(p)

    # Remove duplicates and limit length
    seen = set()
    result = []
    for x in out:
        key = x.strip()
        if key not in seen:
            seen.add(key)
            result.append(key)

    result = [x for x in result if len(x) <= 20]
    return ";".join(result)

# Apply mapping to harmonize dataset names
df_mismatch["Data Source Screening"] = df_mismatch["Data Source Screening"].apply(map_partial_sources)
df_mismatch["Data Source Sensitivity"] = df_mismatch["Data Source Sensitivity"].apply(map_partial_sources)
df_mismatch

,DOI,Data Source Screening,Data Source Sensitivity
1,10.1145/3581783.3613774,RFSJ,AQA-7;BPAD;FineDiving;FIS-V;Fitness-AQA;MIT-Di...
3,10.1155/2023/3649217,,
5,10.1109/TVCG.2023.3247092,3D Animation;JDM,JDM
6,10.1109/WACV.2019.00161,AQA-7,AQA-7
9,10.1007/978-3-319-10599-4_36,MIT-Dive;MIT-Skate,MIT-Dive;MIT-Skate
10,10.1007/s00521-023-09068-w,AQA-7;JIGSAWS;MTL-AQA,AQA-7;JIGSAWS;MTL-AQA
12,10.1007/978-3-031-19839-7_7,Fitness-AQA,Fitness-AQA;Waseda Backsquat
13,10.1109/TPAMI.2024.3378753,EHE;KIMORE;UI-PRMD,EHE;KIMORE
14,10.1109/TIP.2023.3331212,AQA-7;FineDiving;MTL-AQA,AQA-7;AQA-7-Dive;FineDiving;MIT-Dive;MTL-AQA;U...
15,10.1109/CVPR52688.2022.00296,FineDiving,FineDiving;MIT-Skate;MTL-AQA;MIT-Dive


In [39]:
df_mismatch_2 = df_mismatch[
    df_mismatch["Data Source Sensitivity"].notna()
    & (df_mismatch["Data Source Sensitivity"].astype(str).str.strip() != "")
    & df_mismatch["Data Source Screening"].notna()
    & (df_mismatch["Data Source Screening"].astype(str).str.strip() != "")
    & (
        df_mismatch["Data Source Screening"]
        != df_mismatch["Data Source Sensitivity"]
    )
    & (
        df_mismatch["Data Source Sensitivity"].astype(str).str.len()
        >= df_mismatch["Data Source Screening"].astype(str).str.len()
    )
    & (
        df_mismatch["Data Source Sensitivity"]
        .astype(str)
        .str.split(";")
        .apply(lambda x: len([s.strip() for s in x if s.strip()]) <= 3)
    )
].copy()


def missing_from_screening(row):
    sensitivity = [
        s.strip()
        for s in str(row["Data Source Sensitivity"]).split(";")
        if s.strip()
    ]

    screening = {
        s.strip()
        for s in str(row["Data Source Screening"]).split(";")
        if s.strip()
    }

    missing = [
        s for s in sensitivity
        if s not in screening
    ]

    return "; ".join(missing)


df_mismatch_2["Sensitivity not in Screening"] = df_mismatch_2.apply(
    missing_from_screening,
    axis=1
)

df_mismatch_2

,DOI,Data Source Screening,Data Source Sensitivity,Sensitivity not in Screening
12,10.1007/978-3-031-19839-7_7,Fitness-AQA,Fitness-AQA;Waseda Backsquat,Waseda Backsquat
19,10.1109/CBMS61543.2024.00016,Fitness-AQA,AQA-7;Fitness-AQA,AQA-7
29,10.1109/TMM.2023.3328180,OlympicFS,FineDiving;OlympicFS,FineDiving
39,10.1145/3622896.3622916,FineDiving,AQA-7;FineDiving;MTL-AQA,AQA-7; MTL-AQA
47,10.1109/TCSVT.2022.3143549,MTL-AQA;Rhythmic Gymnastics,JIGSAWS;MTL-AQA;Rhythmic Gymnastics,JIGSAWS
50,10.1016/j.jvcir.2022.103625,FIS-V;MIT-Skate,ExPose;FIS-V;MIT-Skate,ExPose
53,10.1109/MMSP55362.2022.9949464,TaiChi-24,FineDiving;TaiChi-24,FineDiving
58,10.1109/CVPR42600.2020.00986,AQA-7;MTL-AQA,AQA-7;JIGSAWS;MTL-AQA,JIGSAWS
61,10.1109/CVPR.2019.00039,MTL-AQA,MIT-Dive;MTL-AQA;UNLV-Dive,MIT-Dive; UNLV-Dive
63,10.1109/ICASSP48485.2024.10448158,TAQR,JIGSAWS;TAQR,JIGSAWS
